# r3con — provenance

**TL;DR** — r3con records where an answer came from:

- the answer cites the relevance snippets and the parsed records it used, separately
  (`result.provenance["summary_ids"]` and `result.provenance["parse_ids"]`);
- each relevance snippet cites the other documents' snippets it used (`result.relevance_provenance`);
- each document's parse cites the snippets it used (`result.parsing_provenance`);
- each parsed record names its document (`"document"` in `result.structured_context`).

They cite documents by id, a part of a document read in parts as `<id>.<k>`, and
`result.document_ids[i]` is the id of `documents[i]`.
The memos and the question are the [quickstart](quickstart.ipynb)'s.

In [1]:
import os
from pathlib import Path

from r3con import r3con

# The key comes from the provider's own variable, or a .env beside this notebook.
assert os.environ.get("OPENAI_API_KEY") or os.path.exists(".env"), (
    "Set OPENAI_API_KEY first."
)

MODEL = "openai/gpt-6-luna"  # any litellm model string

QUESTION = (
    "Which maintenance contractor's sites logged the most equipment incidents in Q3 "
    "in total, and how many? Give the contractor's name, not its code."
)

In [2]:
docs = r3con.read_documents("memos")
result = r3con.run(question=QUESTION, documents=docs, model=MODEL)
print(result)

Halloran Services Ltd’s sites logged the most equipment incidents in Q3, with 11 in total (5 at Northgate and 6 at Riverside).


In [3]:
# read_documents reads a folder in sorted path order, so names[i] is docs[i]'s file.
names = [p.name for p in sorted(Path("memos").glob("*.txt"))]
name_of = dict(zip(result.document_ids, names, strict=True))

# a part of a document read in parts is cited as <id>.<k>
summaries = [name_of[id_.split(".")[0]] for id_ in result.provenance["summary_ids"]]
parses = [name_of[id_] for id_ in result.provenance["parse_ids"]]

print("ids                 :", name_of)
print("answer's summaries  :", summaries)
print("answer's parses     :", parses)

ids                 : {'7ae9ad': '01_northgate.txt', '196a81': '02_eastfield.txt', 'e8d564': '03_riverside.txt', '6385ad': '04_contractor_registry.txt', '1f6020': '05_cafeteria.txt'}
answer's summaries  : ['01_northgate.txt', '03_riverside.txt', '02_eastfield.txt', '04_contractor_registry.txt']
answer's parses     : ['01_northgate.txt', '03_riverside.txt', '02_eastfield.txt', '04_contractor_registry.txt']


In [4]:
summarized = [ref.split(".")[0] for ref in result.provenance["summary_ids"]]
for id_ in dict.fromkeys(summarized):
    i = result.document_ids.index(id_)
    summary = result.relevant_context[i].strip().partition("\n")[0]
    drew_on = [name_of[ref.split(".")[0]] for ref in result.relevance_provenance[i]]
    print(f"{name_of[id_]}\n  summary : {summary[:100]}\n  drew on : {drew_on}\n")

01_northgate.txt
  summary : A Northgate site Q3 operations memo reporting 5 equipment incidents and planned maintenance by contr
  drew on : ['04_contractor_registry.txt']

03_riverside.txt
  summary : A Riverside Q3 operations memo reports 6 equipment incidents under contractor code CT-118, which the
  drew on : ['04_contractor_registry.txt']

02_eastfield.txt
  summary : An Eastfield Q3 operations memo reports 9 equipment incidents at the site, where maintenance is perf
  drew on : ['04_contractor_registry.txt']

04_contractor_registry.txt
  summary : An approved contractor-registry extract that provides the authoritative name-to-code mapping: CT-118
  drew on : ['01_northgate.txt', '02_eastfield.txt', '03_riverside.txt']



In [5]:
for field, records in result.structured_context.items():
    for record in records:
        if record.get("document") in result.provenance["parse_ids"]:
            print(f"{name_of[record['document']]}  {field}: {record}")

01_northgate.txt  site_incident_reports: {'document': '7ae9ad', 'site': 'Northgate', 'period': 'Q3', 'equipment_incident_count': 5, 'contractor_code': 'CT-118'}
02_eastfield.txt  site_incident_reports: {'document': '196a81', 'site': 'Eastfield', 'period': 'Q3', 'equipment_incident_count': 9, 'contractor_code': 'CT-204'}
03_riverside.txt  site_incident_reports: {'document': 'e8d564', 'site': 'Riverside', 'period': 'Q3', 'equipment_incident_count': 6, 'contractor_code': 'CT-118'}
04_contractor_registry.txt  contractor_mappings: {'document': '6385ad', 'contractor_code': 'CT-118', 'contractor_name': 'Halloran Services Ltd'}
04_contractor_registry.txt  contractor_mappings: {'document': '6385ad', 'contractor_code': 'CT-204', 'contractor_name': 'Merrow Facilities Group'}
04_contractor_registry.txt  contractor_mappings: {'document': '6385ad', 'contractor_code': 'CT-311', 'contractor_name': 'Bellweather Grounds'}


In the run folder, `result.run_dir`, the same ids are in `ids.json` (id → index in `documents`), `relevance/result.json` (each round's relevance snippets and what they cite), `structuring/parsing/result.json` (the records and what each document's parse cites) and `reasoning/result.json` (the answer, and the snippets and parses it cites).